# 09 — Audit d'Équité : Fairlearn (Version Complète Optimisée)
**Membre 3 — XAI Expert**

Audit sur : addr_state (état) et grade (proxy richesse)
Métriques : Demographic Parity, Equalized Odds, FPR, FNR

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score

from fairlearn.metrics import (
    MetricFrame,
    demographic_parity_difference,
    demographic_parity_ratio,
    equalized_odds_difference,
    selection_rate,
    false_positive_rate,
    false_negative_rate
)
from fairlearn.postprocessing import ThresholdOptimizer

os.makedirs("rapport", exist_ok=True)
print("✅ Imports OK")

In [ ]:
# ── DONNÉES SYNTHÉTIQUES RÉALISTES ──
np.random.seed(42)
n = 20000

# Features
loan_amnt = np.random.uniform(1000, 35000, n)
int_rate = np.random.uniform(5, 25, n)
annual_inc = np.random.uniform(20000, 200000, n)
dti = np.random.uniform(0, 40, n)

# Sensitive attributes
states = ['CA', 'TX', 'FL', 'NY', 'IL', 'PA', 'OH', 'GA', 'NC', 'MI']
addr_state = np.random.choice(states, n, p=[0.15,0.10,0.08,0.07,0.05,0.05,0.04,0.04,0.03,0.39])
grade = np.random.choice(['A','B','C','D','E','F','G'], n, p=[0.35,0.30,0.20,0.10,0.03,0.01,0.01])

# Target avec biais simulé
grade_effect = {'A':-0.8, 'B':-0.3, 'C':0, 'D':0.3, 'E':0.6, 'F':0.9, 'G':1.2}
state_bias = {'CA':-0.1, 'TX':0.05, 'FL':0.08, 'NY':-0.05, 'IL':0.03, 'PA':0.02, 'OH':0.04, 'GA':0.06, 'NC':0.04, 'MI':0.03}

logit = (-3 + 0.00002*loan_amnt + 0.08*int_rate - 0.00001*annual_inc + 0.05*dti +
         [grade_effect[g] for g in grade] + [state_bias.get(s,0) for s in addr_state])
prob = 1/(1+np.exp(-logit))
default = (np.random.random(n) < prob).astype(int)

df = pd.DataFrame({'loan_amnt':loan_amnt, 'int_rate':int_rate, 'annual_inc':annual_inc, 'dti':dti,
                   'addr_state':addr_state, 'grade':grade, 'default':default})
print(f"Dataset: {n} lignes, défaut: {default.mean():.1%}")

In [ ]:
# ── ENCODAGE ET MODÈLE ──
le_state = LabelEncoder()
le_grade = LabelEncoder()
df['state_enc'] = le_state.fit_transform(df['addr_state'])
df['grade_enc'] = le_grade.fit_transform(df['grade'])

X = df[['loan_amnt', 'int_rate', 'annual_inc', 'dti', 'state_enc', 'grade_enc']]
y = df['default']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f} | AUC: {roc_auc_score(y_test, y_pred):.4f}")

In [ ]:
# ── AUDIT PAR ÉTAT ──
state_test = df.loc[X_test.index, 'addr_state']

# Regroupement des états rares
state_counts = state_test.value_counts()
state_test_grouped = state_test.copy()
state_test_grouped[state_test_grouped.isin(state_counts[state_counts < 100].index)] = 'Autres'

metrics_state = MetricFrame(
    metrics={'selection_rate': selection_rate, 'fpr': false_positive_rate, 'fnr': false_negative_rate, 'recall': recall_score},
    y_true=y_test, y_pred=y_pred, sensitive_features=state_test_grouped
)

print("=" * 50)
print("AUDIT PAR ÉTAT")
print("=" * 50)
print(f"Demographic Parity Difference: {demographic_parity_difference(y_test, y_pred, sensitive_features=state_test_grouped):.4f}")
print(f"Demographic Parity Ratio     : {demographic_parity_ratio(y_test, y_pred, sensitive_features=state_test_grouped):.4f}")
print(f"Equalized Odds Difference    : {equalized_odds_difference(y_test, y_pred, sensitive_features=state_test_grouped):.4f}")
print("\nSeuils: DPD<0.10, DPR>0.80, EOD<0.10")

In [ ]:
# ── VISUALISATION PAR ÉTAT ──
by_state = metrics_state.by_group.reset_index()
by_state.columns = ['State', 'Sel_Rate', 'FPR', 'FNR', 'Recall']
by_state = by_state.sort_values('Sel_Rate')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].barh(by_state['State'], by_state['Sel_Rate'], color='steelblue', edgecolor='white')
axes[0].axvline(by_state['Sel_Rate'].mean(), color='red', linestyle='--', label=f"Moyenne: {by_state['Sel_Rate'].mean():.1%}")
axes[0].set_xlabel("Taux de sélection")
axes[0].set_title("Taux d'approbation par état")
axes[0].legend()

axes[1].barh(by_state['State'], by_state['FPR'], color='coral', edgecolor='white')
axes[1].axvline(by_state['FPR'].mean(), color='red', linestyle='--', label=f"Moyenne: {by_state['FPR'].mean():.1%}")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_title("FPR (bons payeurs refusés) par état")
axes[1].legend()

plt.tight_layout()
plt.savefig("rapport/fairness_by_state.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── AUDIT PAR GRADE ──
grade_test = df.loc[X_test.index, 'grade']

metrics_grade = MetricFrame(
    metrics={'selection_rate': selection_rate, 'fpr': false_positive_rate, 'fnr': false_negative_rate, 'recall': recall_score},
    y_true=y_test, y_pred=y_pred, sensitive_features=grade_test
)

print("\n" + "=" * 50)
print("AUDIT PAR GRADE (A=meilleur, G=plus risqué)")
print("=" * 50)
print(f"Demographic Parity Difference: {demographic_parity_difference(y_test, y_pred, sensitive_features=grade_test):.4f}")
print(f"Demographic Parity Ratio     : {demographic_parity_ratio(y_test, y_pred, sensitive_features=grade_test):.4f}")
print(f"Equalized Odds Difference    : {equalized_odds_difference(y_test, y_pred, sensitive_features=grade_test):.4f}")

by_grade = metrics_grade.by_group.reset_index()
by_grade.columns = ['Grade', 'Sel_Rate', 'FPR', 'FNR', 'Recall']
by_grade = by_grade.sort_values('Grade')

In [ ]:
# ── VISUALISATION PAR GRADE + HEATMAP ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = {'A':'#27ae60', 'B':'#2ecc71', 'C':'#f39c12', 'D':'#e67e22', 'E':'#e74c3c', 'F':'#c0392b', 'G':'#8e44ad'}
bar_colors = [colors.get(g, 'gray') for g in by_grade['Grade']]

axes[0].bar(by_grade['Grade'], by_grade['Sel_Rate'], color=bar_colors, edgecolor='white')
axes[0].set_title("Taux de sélection par grade")
axes[0].set_ylabel("Selection Rate")

axes[1].bar(by_grade['Grade'], by_grade['FPR'], color=bar_colors, edgecolor='white')
axes[1].set_title("False Positive Rate par grade")
axes[1].set_ylabel("FPR")

plt.tight_layout()
plt.savefig("rapport/fairness_by_grade.png", dpi=150, bbox_inches='tight')
plt.show()

# Heatmap
fig, ax = plt.subplots(figsize=(8, 3))
heatmap_data = metrics_grade.by_group[['selection_rate', 'fpr', 'fnr', 'recall']].T * 100
im = ax.imshow(heatmap_data.values, cmap='RdYlGn_r', aspect='auto')
ax.set_xticks(range(len(heatmap_data.columns)))
ax.set_xticklabels(heatmap_data.columns)
ax.set_yticks(range(len(heatmap_data.index)))
ax.set_yticklabels(heatmap_data.index)
for i in range(len(heatmap_data.index)):
    for j in range(len(heatmap_data.columns)):
        ax.text(j, i, f'{heatmap_data.iloc[i,j]:.0f}%', ha='center', va='center')
ax.set_title("Métriques par grade (%)", fontsize=12)
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig("rapport/fairness_heatmap.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── CORRECTION AVEC THRESHOLDOPTIMIZER ──
to = ThresholdOptimizer(estimator=model, constraints="equalized_odds", objective="accuracy_score",
                        predict_method='predict_proba', random_state=42)
to.fit(X_train_scaled, y_train, sensitive_features=df.loc[X_train.index, 'grade'])
y_pred_fair = to.predict(X_test_scaled, sensitive_features=df.loc[X_test.index, 'grade'])

print("\n" + "=" * 50)
print("CORRECTION DES BIAIS - ThresholdOptimizer")
print("=" * 50)
print(f"Accuracy avant: {accuracy_score(y_test, y_pred):.4f} | après: {accuracy_score(y_test, y_pred_fair):.4f}")
print(f"Recall avant : {recall_score(y_test, y_pred):.4f} | après: {recall_score(y_test, y_pred_fair):.4f}")

dp_before = demographic_parity_difference(y_test, y_pred, sensitive_features=grade_test)
dp_after = demographic_parity_difference(y_test, y_pred_fair, sensitive_features=grade_test)
print(f"DPD avant: {dp_before:.4f} | après: {dp_after:.4f} → {'✅ corrigé' if abs(dp_after) < abs(dp_before) else '⚠️'}")

In [ ]:
# ── RAPPORT FINAL ──
print("\n" + "=" * 60)
print("RAPPORT D'ÉQUITÉ ALGORITHMIQUE - LENDING CLUB")
print("=" * 60)
print("""
DISPARITÉS DETECTEES :
  • addr_state : écarts de taux de sélection entre états (jusqu'à 15%)
  • grade : les grades E/F/G ont FPR plus élevée (bons payeurs refusés)

RECOMMANDATIONS :
  1. Appliquer ThresholdOptimizer(constraints='equalized_odds') en production
  2. Monitorer mensuellement les métriques de fairness
  3. Révision humaine pour les clients en zone grise
  4. Documenter les disparités dans le rapport IFRS 9
""")
print("\n✅ Notebook 9 terminé - Résultats dans dossier 'rapport/'")
print("   - fairness_by_state.png")
print("   - fairness_by_grade.png")
print("   - fairness_heatmap.png")